# Тест Хаусмана
 реализация эконометрического теста Хаусмана для панельных данных в виде класса.
 
  Тест позволяет выбрать между моделью с фиксированными эффектами (FE) и моделью со случайными эффектами (RE) с помощью некоторых методов библиотеки `linearmodels`.

In [66]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects

In [67]:
class HausmanTest:
    def __init__(self, df, dependent_var, independent_vars):
        """
        :param df: pandas DataFrame с панельными данными
        :param dependent_var: строка, название зависимой переменной
        :param independent_vars: список строк, названия независимых переменных
        """
        self.df = df.copy()
        self.dependent_var = dependent_var
        self.independent_vars = independent_vars
        self.fe_model = None
        self.re_model = None
        self.fe_results = None
        self.re_results = None

        if not isinstance(self.df.index, pd.MultiIndex):
            self.df = self.df.set_index(['region', 'year'])
        
    def fit_models(self):
        y = self.df[self.dependent_var]
        X = self.df[self.independent_vars]
        X = sm.add_constant(X)
        
        # модель с фикс эффектом
        self.fe_model = PanelOLS(y, X, entity_effects=True)
        self.fe_results = self.fe_model.fit()
        
        # модель с рандомным эффектом
        self.re_model = RandomEffects(y, X)
        self.re_results = self.re_model.fit()
        
    def perform_test(self):
        if self.fe_results is None or self.re_results is None:
            self.fit_models()
            
        # извлекаем коэффициенты, исключая константу (т.к. FE поглощается индвивидуальными эффектами регионов (в нашем случае)
        b_fe = self.fe_results.params.drop('const', errors='ignore')
        b_re = self.re_results.params.drop('const', errors='ignore')
        
        v_fe = self.fe_results.cov.drop('const', axis=0, errors='ignore').drop('const', axis=1, errors='ignore')
        v_re = self.re_results.cov.drop('const', axis=0, errors='ignore').drop('const', axis=1, errors='ignore')
        
        b_diff = b_fe - b_re
        df_diff = len(b_diff) # степени свободы = количеству оцениваемых коэффициентов
        v_diff = v_fe - v_re  
        
        # Вычисление статистики Хаусмана
        try:
            inv_v_diff = np.linalg.inv(v_diff.values)
            h_stat = b_diff.values.T @ inv_v_diff @ b_diff.values
        except np.linalg.LinAlgError:
            print("матрица разности ковариаций вырождена. используем псевдообратную .")
            inv_v_diff = np.linalg.pinv(v_diff.values)
            h_stat = b_diff.values.T @ inv_v_diff @ b_diff.values

        p_value = stats.chi2.sf(h_stat, df_diff)
        
        return h_stat, p_value, df_diff
        
    def print_results(self):
        h_stat, p_value, df_diff = self.perform_test()
        
        print(f"Результаты теста Хаусмана")
        print(f"статистика Хаусмана (хи-квадрат): {h_stat:.4f}")
        print(f"df: {df_diff}")
        print(f"p-value: {p_value:.4f}\n")
        
        alpha = 0.05
        if p_value < alpha:
            print("Отвергаем нулевую гипотезу (H0).")
            print("Вывод: Оценки модели со случайными эффектами (RE) несостоятельны.")
            print("Предпочтительно использовать модель с фиксированными эффектами (FE).")
        else:
            print("Нет оснований отвергнуть нулевую гипотезу (H0).")
            print("Вывод: Оценки модели со случайными эффектами (RE) состоятельны и эффективны.")
            print("Предпочтительно использовать модель со случайными эффектами (RE).")

In [68]:
df = pd.read_csv('cleaned_df.csv')
df

,region,year,"Численность постоянного населения на 1 число, тыс. человек","Численность постоянного населения в среднем за год, тыс. человек","Общий демографический прирост (-убыль), человек","Ожидаемая продолжительность жизни, лет","Естественный прирост населения, человек","Коэффициент естественного прироста населения (на 1000 населения), человек","Число родившихся, человек","Коэффициент рождаемости (на 1000 населения), человек",...,"Бюджетная обеспеченность, %","Расходы консолидированных бюджетов субъектов ДФО, млрд рублей","Расходы консолидированных бюджетов субъектов ДФО, в % к СППГ","Обеспечение расходов налоговыми и неналоговыми доходами,%","Дефицит (профицит) консолидированные бюджетов субъектов ДФО, млрд рублей","Государственный долг субъектов ДФО, млрд рублей","Отношение объема гос. долга к собственным доходам консолидированного бюджета, %","Государственный и муниципальный долг субъектов ДФО, млрд рублей","Государственный и муниципальный долг субъектов ДФО, в % к СППГ","Объем экспорта, млн долл."
0,Амурская область,2010,828.660,831.783,-4942,64.400,-1261,-1.50,11479.0,13.8,...,122.345815,45.762300,112.353389,121.417039,-0.347400,6.741365,12.132781,7.296904,107.458884,158.800000
1,Амурская область,2011,820.627,824.644,-7087,64.810,-1000,-1.20,11211.0,13.6,...,121.046454,54.375500,118.821606,112.480785,-3.847800,11.558795,18.898658,12.533565,171.765508,228.600000
2,Амурская область,2012,815.163,817.895,-4663,65.080,-340,-0.40,11740.0,14.4,...,66.539844,58.567700,107.709722,59.075773,-6.569800,15.165396,43.831483,16.307122,130.107613,407.500000
3,Амурская область,2013,808.726,811.944,-5636,66.320,133,0.20,11453.0,14.1,...,47.087610,78.530400,134.084828,45.400787,-2.813200,22.815829,63.993381,24.346617,149.300514,446.310465
4,Амурская область,2014,806.524,807.625,-1401,66.910,-136,-0.20,11094.0,13.7,...,68.513080,65.427200,83.314487,56.314758,-11.648900,28.227767,76.611851,30.000629,123.222988,383.344995
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,Чукотский автономный округ,2020,47.583,48.066,-736,65.000,24,0.50,546.0,11.4,...,45.987793,49.872871,88.578885,50.102725,4.462563,10.249343,41.017605,10.520343,106.918103,288.844819
191,Чукотский автономный округ,2021,47.906,47.745,513,64.000,-38,-0.80,502.0,10.5,...,42.411445,55.948689,112.182610,40.832967,-2.082309,9.682713,42.383438,9.897213,94.076903,281.685323
192,Чукотский автономный округ,2022,47.840,47.873,-66,66.200,18,0.40,508.0,10.6,...,39.390280,58.328805,104.254105,35.998900,-5.021927,8.881964,42.299643,9.219564,93.153139,109.348459
193,Чукотский автономный округ,2023,48.029,47.935,189,66.560,39,0.80,523.0,10.9,...,58.398439,1.266134,121.355089,151.214479,2.012340,8.999306,470.041409,9.454448,74.388565,109.348459


In [69]:
df.describe()

,year,"Численность постоянного населения на 1 число, тыс. человек","Численность постоянного населения в среднем за год, тыс. человек","Общий демографический прирост (-убыль), человек","Ожидаемая продолжительность жизни, лет","Естественный прирост населения, человек","Коэффициент естественного прироста населения (на 1000 населения), человек","Число родившихся, человек","Коэффициент рождаемости (на 1000 населения), человек","Число умерших, человек",...,"Бюджетная обеспеченность, %","Расходы консолидированных бюджетов субъектов ДФО, млрд рублей","Расходы консолидированных бюджетов субъектов ДФО, в % к СППГ","Обеспечение расходов налоговыми и неналоговыми доходами,%","Дефицит (профицит) консолидированные бюджетов субъектов ДФО, млрд рублей","Государственный долг субъектов ДФО, млрд рублей","Отношение объема гос. долга к собственным доходам консолидированного бюджета, %","Государственный и муниципальный долг субъектов ДФО, млрд рублей","Государственный и муниципальный долг субъектов ДФО, в % к СППГ","Объем экспорта, млн долл."
count,195.000000,195.000000,195.000000,195.000000,195.000000,1.950000e+02,195.000000,1.950000e+02,195.000000,1.950000e+02,...,195.000000,195.000000,195.000000,195.000000,195.000000,195.000000,195.000000,195.000000,195.000000,195.000000
mean,2017.000000,12489.276338,12489.371492,-7097.989744,68.135385,-2.319745e+04,-0.296615,1.446568e+05,12.322838,1.652567e+05,...,66.582073,879.833078,109.684332,65.797884,-7.934184,189.570445,119.334624,218.438296,119.881464,36921.739384
std,4.331615,38733.385276,38723.760290,90611.900397,2.446364,1.178333e+05,3.400932,4.473650e+05,2.302174,5.151942e+05,...,35.197609,2917.812358,11.053152,36.926652,98.912096,583.375732,273.807689,671.384913,60.592312,115167.535115
min,2010.000000,47.583000,47.745000,-613439.000000,57.500000,-1.043341e+06,-9.100000,5.020000e+02,7.500000,4.540000e+02,...,16.032096,0.830000,74.895407,14.581735,-676.569556,0.000000,0.000000,0.129673,34.282416,8.700000
25%,2013.000000,292.913000,293.861500,-6012.000000,66.875000,-2.132500e+03,-2.300000,3.278000e+03,10.600000,3.594000e+03,...,52.630151,31.764325,104.484936,49.659487,-5.056856,5.002237,17.767996,6.830537,96.576854,377.393330
50%,2017.000000,968.065000,967.159000,-2101.000000,68.420000,-2.450000e+02,-0.600000,1.199700e+04,12.300000,1.104600e+04,...,64.938443,65.427200,109.764944,63.179326,-1.112200,12.128367,30.873348,14.180010,107.458884,977.053298
75%,2021.000000,1336.326500,1337.654000,-387.500000,69.785000,5.325000e+02,1.650000,1.751100e+04,13.800000,1.891500e+04,...,77.092515,157.078713,114.769213,74.378384,0.632939,35.236378,58.908177,37.696436,121.932552,4720.998757
max,2024.000000,147959.284000,147899.994000,319872.000000,73.550000,3.203800e+04,9.200000,1.942683e+06,17.800000,2.441594e+06,...,493.402568,19626.301060,160.651726,493.883275,660.761442,3186.956508,1604.220417,3563.468653,756.985473,525976.302260


In [70]:
df.columns.tolist()

['region',
 'year',
 'Численность постоянного населения на 1 число, тыс. человек',
 'Численность постоянного населения в среднем за год, тыс. человек',
 'Общий демографический прирост (-убыль), человек',
 'Ожидаемая продолжительность жизни, лет ',
 'Естественный прирост населения, человек',
 'Коэффициент естественного прироста населения (на 1000 населения), человек',
 'Число родившихся, человек',
 'Коэффициент рождаемости (на 1000 населения), человек',
 'Число умерших, человек',
 'Коэффициент смертности (на 1000 населения), человек',
 'Миграционный прирост населения, человек ',
 'Коэффициент миграционного прироста населения (на 1000 населения), человек',
 'Число прибывших, человек',
 'Число выбывших, человек',
 'Численность занятых (в возрасте 15-72 лет), тыс. человек',
 'Численность безработных (в возрасте 15-72 лет), тыс. Человек',
 'Среднегодовая численность занятых в экономике (расчеты на основе интеграции данных), тыс. человек',
 'Уровень общей безработицы (по методологии МОТ) (в 

In [71]:
y = 'Первичный рынок. Индекс цен на рынке жилья (все типы квартир)'  
xs = ['Среднемесячная номинальная начисленная заработная плата, в % к СППГ', 'Коэффициент естественного прироста населения (на 1000 населения), человек']        

In [72]:

hausman = HausmanTest(df, y, xs)
hausman.print_results()

Результаты теста Хаусмана
статистика Хаусмана (хи-квадрат): 5.1056
df: 2
p-value: 0.0779

Неn оснований отвергнуть нулевую гипотезу (H0).
Вывод: Оценки модели со случайными эффектами (RE) состоятельны и эффективны.
Предпочтительно использовать модель со случайными эффектами (RE).
